# 06 · Descarga de datos

**Esta es la única notebook que escribe archivos:** crea `data/manifest.csv` y, si `EXPORTAR = True`, envía las exportaciones a Google Drive.

**Flujo:**
1. Revisar el plan (sección 2).
2. Poner `EXPORTAR = True` y ejecutar la sección 3. Las tareas aparecen en la pestaña **Tasks** de code.earthengine.google.com.
3. Cuando terminen, copiar los `.tif` de Google Drive a `data/raw/gee/<carpeta>/`. Si tienes Google Drive para escritorio, la carpeta `GEE_exports` se sincroniza sola.
4. Ejecutar la sección 4 para verificar cada archivo y guardar su MD5 en el manifest.

**Formato de los archivos:** reflectancia × 10000 en Int16 (dividir entre 10000 al leer). El conteo de observaciones `n_obs` va como última banda.

**Estado:** notebook nuevo, todavía sin ejecutar. Con `EXPORTAR = False` no se envía nada.


## 1. Configuración


In [ ]:
from datetime import date, datetime
from pathlib import Path
import hashlib
import io
import json
import re

import ee
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

GEE_PROJECT = "viirs-peru"
ANIO_INI, ANIO_FIN = 2010, 2025  # 15 periodos anuales: 2010 -> 2025

# Bandas por sensor: nombre común -> banda original
BANDAS_LANDSAT = {
    "L5": {"blue": "SR_B1", "green": "SR_B2", "red": "SR_B3", "nir": "SR_B4", "swir1": "SR_B5", "swir2": "SR_B7"},
    "L7": {"blue": "SR_B1", "green": "SR_B2", "red": "SR_B3", "nir": "SR_B4", "swir1": "SR_B5", "swir2": "SR_B7"},
    "L8": {"blue": "SR_B2", "green": "SR_B3", "red": "SR_B4", "nir": "SR_B5", "swir1": "SR_B6", "swir2": "SR_B7"},
    "L9": {"blue": "SR_B2", "green": "SR_B3", "red": "SR_B4", "nir": "SR_B5", "swir1": "SR_B6", "swir2": "SR_B7"},
}
BANDAS_S2 = {"blue": "B2", "green": "B3", "red": "B4", "nir": "B8", "swir1": "B11", "swir2": "B12"}
COLECCIONES_LANDSAT = {
    "L5": "LANDSAT/LT05/C02/T1_L2",
    "L7": "LANDSAT/LE07/C02/T1_L2",
    "L8": "LANDSAT/LC08/C02/T1_L2",
    "L9": "LANDSAT/LC09/C02/T1_L2",
}
COLECCION_S2 = "COPERNICUS/S2_SR_HARMONIZED"
MB_ASSET = "projects/mapbiomas-public/assets/peru/collection4/mapbiomas_peru_collection4_coverage_v1"


def buscar_raiz(inicio=None):
    """Sube por las carpetas hasta encontrar la raíz del proyecto (la que tiene el límite provincial).
    Se usa el GeoPackage como marcador porque puede existir una carpeta data/raw dentro de notebooks/."""
    inicio = Path(inicio or Path.cwd()).resolve()
    for carpeta in [inicio, *inicio.parents]:
        if (carpeta / "data" / "raw" / "Provincia" / "PROVINCIA.gpkg").is_file():
            return carpeta
    raise FileNotFoundError("No encuentro data/raw/Provincia/PROVINCIA.gpkg desde esta ubicación.")


def iniciar_gee(proyecto=GEE_PROJECT):
    try:
        ee.Initialize(project=proyecto)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=proyecto)


def rango_fechas(col):
    """Primera y última fecha de una colección, como texto."""
    d = ee.Dictionary({
        "min": col.aggregate_min("system:time_start"),
        "max": col.aggregate_max("system:time_start"),
    }).getInfo()
    if d["min"] is None:
        return "sin fecha"
    a = pd.to_datetime(d["min"], unit="ms").strftime("%Y-%m-%d")
    b = pd.to_datetime(d["max"], unit="ms").strftime("%Y-%m-%d")
    return f"{a} → {b}"


def md5(ruta):
    """Suma MD5 de un archivo, leído por bloques."""
    h = hashlib.md5()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()


iniciar_gee()
RAIZ = buscar_raiz()
GPKG = RAIZ / "data" / "raw" / "Provincia" / "PROVINCIA.gpkg"
gdf = gpd.read_file(GPKG)
prov = gdf[(gdf["ccdd"] == "08") & (gdf["ccpp"] == "01")]
assert len(prov) == 1, "El filtro de provincia no devolvió exactamente 1 registro"
aoi = ee.Geometry(prov.geometry.iloc[0].simplify(0.0001, preserve_topology=True).__geo_interface__)
print("Provincia:", prov["nombprov"].iloc[0], "| ventana:", ANIO_INI, "-", ANIO_FIN)


In [ ]:
MASCARA_LANDSAT_BITS = 31  # bits 0 a 4 de QA_PIXEL: relleno, nube dilatada, cirrus, nube y sombra


def landsat_estandar(img, sensor):
    """Escala a reflectancia (0-1), pone nombres comunes y enmascara relleno, nubes y sombras."""
    b = BANDAS_LANDSAT[sensor]
    libre = img.select("QA_PIXEL").bitwiseAnd(MASCARA_LANDSAT_BITS).eq(0)
    sr = img.select(list(b.values())).multiply(0.0000275).add(-0.2)
    return sr.rename(list(b.keys())).updateMask(libre).copyProperties(img, ["system:time_start"])


def s2_estandar(img):
    """Reflectancia (0-1), nombres comunes y máscara con la capa SCL (nube, sombra, cirrus, sin dato).
    La colección HARMONIZED ya unifica el desfase de 1000 del baseline 04.00. Verificar con el mínimo de una banda."""
    scl = img.select("SCL")
    libre = (scl.neq(0).And(scl.neq(1)).And(scl.neq(3))
             .And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)))
    sr = img.select(list(BANDAS_S2.values())).divide(10000)
    return sr.rename(list(BANDAS_S2.keys())).updateMask(libre).copyProperties(img, ["system:time_start"])


def _conteo_obs(col):
    """Número de observaciones limpias por píxel dentro del año."""
    return col.map(lambda img: img.select("blue").multiply(0).add(1).rename("n_obs")).sum()


def _mapear_sensor(sensor):
    return lambda img: landsat_estandar(img, sensor)


def landsat_anual(anio):
    """Mediana anual (enero a diciembre) de Landsat 5, 7, 8 y 9, con conteo de observaciones limpias."""
    col = None
    for sensor, cid in COLECCIONES_LANDSAT.items():
        parte = (ee.ImageCollection(cid).filterBounds(aoi)
                 .filterDate(f"{anio}-01-01", f"{anio + 1}-01-01")
                 .map(_mapear_sensor(sensor)))
        col = parte if col is None else col.merge(parte)
    return col.median().addBands(_conteo_obs(col)).clip(aoi)


def s2_anual(anio):
    """Mediana anual (enero a diciembre) de Sentinel-2 SR, con conteo de observaciones limpias."""
    col = (ee.ImageCollection(COLECCION_S2).filterBounds(aoi)
           .filterDate(f"{anio}-01-01", f"{anio + 1}-01-01")
           .map(s2_estandar))
    return col.median().addBands(_conteo_obs(col)).clip(aoi)


def indices(img):
    """NDVI, EVI y NBR a partir de bandas estandarizadas (vista previa, sin decidir el índice)."""
    ndvi = img.normalizedDifference(["nir", "red"]).rename("NDVI")
    evi = img.expression(
        "2.5 * (NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1)",
        {"NIR": img.select("nir"), "RED": img.select("red"), "BLUE": img.select("blue")},
    ).rename("EVI")
    nbr = img.normalizedDifference(["nir", "swir2"]).rename("NBR")
    return ndvi.addBands(evi).addBands(nbr)


def miniatura(imagen, vis, dims=700):
    """Descarga una vista previa PNG del área de estudio y la devuelve para matplotlib."""
    url = imagen.getThumbURL({**vis, "region": aoi, "dimensions": dims, "format": "png"})
    resp = requests.get(url, timeout=180)
    resp.raise_for_status()
    return plt.imread(io.BytesIO(resp.content))


In [ ]:
from datetime import datetime

EXPORTAR = False                 # True para enviar las tareas a Google Drive
DRIVE_FOLDER = "GEE_exports"
CARPETA_GEE = RAIZ / "data" / "raw" / "gee"
MANIFEST = RAIZ / "data" / "manifest.csv"
BANDAS_REFL = ["blue", "green", "red", "nir", "swir1", "swir2"]


## 2. Plan de archivos

Cada fila es un archivo que se va a descargar. El nombre sigue el patrón `<fuente>_<producto>_<año>_<resolución>.tif`.


In [ ]:
def construir_imagen(fila):
    """Devuelve la imagen de GEE lista para exportar, según la fila del plan."""
    fuente, anio = fila["fuente"], fila["anio"]
    if fuente == "landsat":
        img = landsat_anual(anio)
        return (img.select(BANDAS_REFL).multiply(10000).round().toInt16()
                .addBands(img.select("n_obs").toInt16()))
    if fuente == "sentinel2":
        img = s2_anual(anio)
        return (img.select(BANDAS_REFL).multiply(10000).round().toInt16()
                .addBands(img.select("n_obs").toInt16()))
    if fuente == "mapbiomas":
        return ee.Image(MB_ASSET).select(f"classification_{anio}").clip(aoi).rename("clase").toInt16()
    if fuente == "worldcover":
        return ee.ImageCollection(fila["coleccion"]).mosaic().clip(aoi).select("Map").rename("clase").toInt16()
    if fuente == "srtm":
        return ee.Image("USGS/SRTMGL1_003").clip(aoi).rename("elevacion").toInt16()
    raise ValueError(f"Fuente desconocida: {fuente}")


plan = []
for anio in range(2010, 2026):
    plan.append({"fuente": "landsat", "carpeta": "landsat", "anio": anio, "resolucion_m": 30,
                 "archivo": f"LS_anual_{anio}_30m.tif",
                 "coleccion": "Landsat C2 L2 (L5, L7, L8, L9)",
                 "bandas": "blue, green, red, nir, swir1, swir2 (x10000) + n_obs",
                 "filtro": "QA_PIXEL sin relleno, nube ni sombra; mediana enero-diciembre"})
    plan.append({"fuente": "mapbiomas", "carpeta": "mapbiomas", "anio": anio, "resolucion_m": 30,
                 "archivo": f"MBC4_cobertura_{anio}_30m.tif",
                 "coleccion": MB_ASSET, "bandas": "clase",
                 "filtro": "recorte a la provincia"})
for anio in range(2019, 2026):
    plan.append({"fuente": "sentinel2", "carpeta": "sentinel2", "anio": anio, "resolucion_m": 10,
                 "archivo": f"S2_anual_{anio}_10m.tif",
                 "coleccion": COLECCION_S2,
                 "bandas": "blue, green, red, nir, swir1, swir2 (x10000) + n_obs",
                 "filtro": "SCL sin nube, sombra, cirrus ni relleno; mediana enero-diciembre"})
plan.append({"fuente": "worldcover", "carpeta": "worldcover", "anio": 2020, "resolucion_m": 10,
             "archivo": "WC-v100_2020_10m.tif", "coleccion": "ESA/WorldCover/v100",
             "bandas": "clase", "filtro": "mosaico recortado"})
plan.append({"fuente": "worldcover", "carpeta": "worldcover", "anio": 2021, "resolucion_m": 10,
             "archivo": "WC-v200_2021_10m.tif", "coleccion": "ESA/WorldCover/v200",
             "bandas": "clase", "filtro": "mosaico recortado"})
plan.append({"fuente": "srtm", "carpeta": "srtm", "anio": None, "resolucion_m": 30,
             "archivo": "SRTM_30m.tif", "coleccion": "USGS/SRTMGL1_003",
             "bandas": "elevacion", "filtro": "recorte a la provincia"})


def cargar_manifest():
    """Lee el manifest si existe y conserva los estados ya registrados."""
    nuevo = pd.DataFrame(plan, dtype=object)
    nuevo["estado"] = "pendiente"
    nuevo["tarea_id"] = ""
    nuevo["fecha_envio"] = ""
    nuevo["ruta_local"] = ""
    nuevo["tamano_mb"] = ""
    nuevo["md5"] = ""
    nuevo["fecha_verificacion"] = ""
    if MANIFEST.exists():
        previo = pd.read_csv(MANIFEST, dtype=str).fillna("")
        nuevo = nuevo.set_index("archivo")
        previo = previo.set_index("archivo")
        comunes = previo.index.intersection(nuevo.index)
        for col in ["estado", "tarea_id", "fecha_envio", "ruta_local", "tamano_mb", "md5", "fecha_verificacion"]:
            if col in previo.columns:
                nuevo.loc[comunes, col] = previo.loc[comunes, col]
        nuevo = nuevo.reset_index()
    return nuevo.astype(object)


manifest = cargar_manifest()
manifest[["archivo", "carpeta", "anio", "resolucion_m", "estado"]]


## 3. Enviar exportaciones a Google Drive

Solo se ejecuta si `EXPORTAR = True`. Cada tarea queda registrada en el manifest con su ID.


In [ ]:
MANIFEST.parent.mkdir(parents=True, exist_ok=True)

if EXPORTAR:
    for i, fila in manifest.iterrows():
        if fila["estado"] == "enviado":
            continue
        fila_plan = next(p for p in plan if p["archivo"] == fila["archivo"])
        stem = Path(fila["archivo"]).stem
        tarea = ee.batch.Export.image.toDrive(
            image=construir_imagen(fila_plan),
            description=stem,
            fileNamePrefix=stem,
            folder=DRIVE_FOLDER,
            region=aoi,
            scale=int(fila["resolucion_m"]),
            maxPixels=1e10,
            fileFormat="GeoTIFF",
            formatOptions={"cloudOptimized": True},
        )
        tarea.start()
        manifest.loc[i, ["estado", "tarea_id", "fecha_envio"]] = [
            "enviado", getattr(tarea, "id", ""), datetime.now().strftime("%Y-%m-%d %H:%M")]
        print("Enviado:", fila["archivo"])
    manifest.to_csv(MANIFEST, index=False)
    print("Manifest guardado en", MANIFEST)
else:
    manifest.to_csv(MANIFEST, index=False)
    print("EXPORTAR = False: no se envió nada. Manifest con el plan en", MANIFEST)


## 4. Verificar descargas en tu carpeta local

Busca cada archivo en `data/raw/gee/<carpeta>/`. Si está, registra tamaño, MD5 y fecha de verificación.


In [ ]:
from datetime import date

for carpeta in ["landsat", "sentinel2", "mapbiomas", "worldcover", "srtm"]:
    (CARPETA_GEE / carpeta).mkdir(parents=True, exist_ok=True)

for i, fila in manifest.iterrows():
    ruta = CARPETA_GEE / fila["carpeta"] / fila["archivo"]
    if ruta.exists():
        manifest.loc[i, ["estado", "ruta_local", "tamano_mb", "md5", "fecha_verificacion"]] = [
            "descargado",
            str(ruta.relative_to(RAIZ)),
            str(round(ruta.stat().st_size / 1e6, 2)),
            md5(ruta),
            date.today().isoformat(),
        ]

manifest.to_csv(MANIFEST, index=False)
print(manifest["estado"].value_counts())


## 5. Resumen

Los archivos que siguen en `enviado` o `pendiente` todavía no están en tu carpeta.


In [ ]:
manifest[["archivo", "carpeta", "anio", "estado", "tamano_mb", "md5"]]
